<a href="https://colab.research.google.com/github/useDeep/learn_ml/blob/main/Next_word_predictor_using_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nltk

In [ ]:
import numpy as np
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize

In [ ]:
data= """Making me inexhaustible
Gives you pleasure;
Exhausting me fully,
You fill me till I am born anew
Taking the little flute that I am,
You cross hills and river banks,
Evoking endless tunes from me!
Who could I tell all this to?
In your everlasting caress
My heart loses sight of limits
And in immense bursts of joy
Song lyrics rise in me.
You give me what I can take
With my hands, night and day,
Years pass but there is no end
To what I receive from you!

When you ask me to sing
My heart swells with pride
As I look intently at you
My eyes moisten with tears
All that is hard and bitter in me
Melts into heavenly music
All my prayers and thoughts
Take wings like merry birds.
You are content with my songs
I know they please you.
They admit me to your company
The One I can’t reach through thought
Accepts me through my songs!
My songs make me forget myself
And let me call my Lord my friend.


O wise one, how do you sing so well?
I listen in amazement, completely enthralled!
Your melodies light up the world
And waft across heavens,
Melting stones, driving everything in the way,
Carrying along with them heavenly music.
Though the tunes keep eluding my voice
I feel like singing in that superb vein
What I would like to say get stuck
And my soul cries out, defeated!
What trap have you ensnared me into?
Your music has me fully in its thrall!

Your touch makes all of me
Feel holy, O Life-giver;
Night and day, I keep you in view
And try to stay pure.
O Truth giver, I try to remind myself,
In everything that I do,
That your presence in my mind
Should guide all my thoughts,
And keep all untruths away.
Because you are in my heart
I can curb all evil and deceit
And check everything hateful—
Because you are rooted in me
My love will bloom and stay pure
Knowing you is my strength
I’ll strive to reveal you in my work.

Just let me sit near you awhile
And indulge me a little
Any work I have left over
I’ll finish later
If I can’t see you
My heart won’t rest
And I’ll drift
In a boundless ocean
Spring murmurs eagerly
At my window this day,
The lazy bee buzzes on
Circling the garden lawn
This is a day for reposing
By myself, of keeping
You always in my view
All I’d like to do this day
Is dedicate my life to you
Singing in quiet repose.

Pluck me this moment
And delay no longer
Let me not lie in the dust forlorn
Let me be part of
The garland you’ll put on
My fate, I keep hoping,
Is to be picked by you alone!
Extract me, pluck me out,
Remove me from all doubt!
Who knows when day will end?
Who knows when night will fall?
Who knows when the time to pray,
And call upon you, may slip away?
Take me, whatever colors I gather,
Choose me, whatever scent I put on,
Make me part of your prayer offerings
Let me serve you any way I can,
Extract me, pluck me out,
Remove me from all doubt!

For you this song has shed
All decoration
For you it has given up
All pretension
Ornaments would block the way
And mar our communion!
They would muffle your words
In needless sounds!
Before you I have no cause
To be vain about my art
O master poet, at your feet I give up
All pride in my craft!
Let me dedicate my life
And make a flute that is plain
Let me fill its stops
With tunes all my own.

The child you dress up like a king
And deck in jewelry,
Weighed down by what he has on,
Loses all pleasure in play
Lest what he wears tears or smudges
Lest what he has on is soiled or stained
He stays away from all company.
The child you dress up like a king
And deck in jewelry
Feels completely fettered!
Why dress your child so, O Mother?
Why deck him in jewelry?
Open the door and let him play,
In sunshine or rain, freely!
The child you dress up like a king
And deck in jewelry,
Will never be part of the world’s festivals
And will never listen in
To the music forever steaming
From the heart of the universe!

I’ll no longer bear on my shoulder
The burden that was me!
I’ll no longer keep myself poor,
Staying indoors endlessly!
Leaving the load I’ve been carrying
At His feet, I’ll go out of the door.
I’ll no longer be what I was
And keep talking needlessly
I’ll no longer bear on my shoulder
The burden that was me!
He will fulfill my desires,
And then part, snuffing his lamp,
In the twinkling of an eye!
I’ll not profane Him by refusing
What He gives with His own hands
I’ll no longer be happy with what rings
Without the music of His love
I’ll no longer bear on my shoulder
The burden that is me!


The steps of your feet sound
Even where the lowliest and poor abound
You have time for all, high or low,
Towards all your love flows.
But when I try to pay my tributes to you
Something holds them back
They fail to reach you where you can be found—
Where the lowliest and poor abound.
Pride can never be part of the places you visit
Where indigent and wretched people abound
You have time for all, high or low
Towards all your love flows
Though you are with the lonely and forsaken
My soul fails to reach out to them
It fails to reach you where you can be found—
Where the lowliest and poor abound."""

In [ ]:
# Tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
device= "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:
# tokenize
tokens= word_tokenize(data.lower())

In [ ]:
# Build the vocabulary
vocab= {'<unk>': 0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]= len(vocab)

vocab_size= len(vocab)
vocab_size

383

In [ ]:
# Extract sentences from the data
sentences= data.split('\n')

In [ ]:
def text_to_indices(sentence, vocab):
  numerical_sentence= []
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [ ]:
# tokenize the words in sentences
numerical_sentences= []
for sentence in sentences:
  numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

numerical_sentences[11:15]

[[52, 53, 54, 39, 2, 55],
 [5, 56, 2, 57, 13, 58, 59],
 [60, 43, 61, 10, 62, 24, 63, 10],
 [64, 65, 66, 67, 68, 69, 70]]

In [ ]:
len(numerical_sentences)

178

In [ ]:
training_sequence= []
for sentence in numerical_sentences:
  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [ ]:
training_sequence[:6]

[[1, 2], [1, 2, 3], [4, 5], [4, 5, 6], [4, 5, 6, 7], [8, 2]]

In [ ]:
len(training_sequence)

946

In [ ]:
# Find the longest lenght about all the training sequences
len_list= 0
for seq in training_sequence:
  if len_list< len(seq):
    len_list= len(seq)

len_list

13

In [ ]:
# Put zeroes in front of all the sequences such that the lenght of each becomes equal to the longest sequence (13 here)

padded_training_sequence= []
for seq in training_sequence:
  padded_training_sequence.append([0]* (len_list - len(seq)) + seq)

padded_training_sequence[: 6]

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 3],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 5],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 5, 6],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 5, 6, 7],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2]]

In [ ]:
padded_training_sequence= torch.tensor(padded_training_sequence, dtype= torch.long)
padded_training_sequence

/tmp/ipykernel_693/156442538.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  padded_training_sequence= torch.tensor(padded_training_sequence, dtype= torch.long)


tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   0,   4,   5],
        ...,
        [  0,   0,   0,  ..., 362,  24, 333],
        [  0,   0,   0,  ...,  24, 333, 363],
        [  0,   0,   0,  ..., 333, 363,  55]])

In [ ]:
padded_training_sequence.shape

torch.Size([946, 13])

In [ ]:
# training data and it's corresponding labels
X= padded_training_sequence[:, : -1]
y= padded_training_sequence[:, -1]

X[:5], y[:5]

(tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 5],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 5, 6]]),
 tensor([2, 3, 5, 6, 7]))

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X= X
    self.y= y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [ ]:
dataset= CustomDataset(X, y)

len(dataset)

946

In [ ]:
dataloader= DataLoader(dataset= dataset,
                       batch_size= 32,
                       shuffle= True)

In [ ]:
next(iter(dataloader))

[tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  32, 256],
         [  0,   0,   0,   0,   0,   0, 118, 119, 104,  10, 120, 121],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 204],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 306],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 224],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   5, 160],
         [  0,   0,   0,   0,   0,  13, 105, 194,  69, 240, 329, 223],
         [  0,   0,   0,   0,   0,   0,  32,  33,  13,  34,  35,  36],
         [  0,   0,   0,   0,   0,   0,   0,   0, 114,   2,  11, 163],
         [  0,   0,   0,   0,   0,   0,   0, 209,  13,  58, 105, 106],
         [  0,   0,   0,   0,   0,   0,   0,   0, 190, 323, 245, 246],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  37,  18],
         [  0,   0,   0,   0,   0,   0,   0,   0,   0, 361,  18, 362],
         [  0,   0,   0,   0,   0,   0, 253,   2,  10, 237,   2, 157],
      

In [ ]:
class LSTMModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self. embedding= nn.Embedding(vocab_size, 100)
    self.lstm= nn.LSTM(input_size= 100,
                       hidden_size= 150,
                       num_layers= 3,
                       batch_first= True,
                      #  bidirectional= True,
                      #  dropout= 0.5,
                       device= device)
    self.linear= nn.Linear(150, vocab_size)

  def forward(self, X):
    embedded= self. embedding(X)
    intermediate_hidden_states, (final_hidden_state, final_cell_state)= self.lstm(embedded)
    result= self.linear(final_hidden_state[-1, :, : ])
    return result

In [ ]:
model= LSTMModel(len(vocab))

In [ ]:
model.to(device)

LSTMModel(
  (embedding): Embedding(383, 100)
  (lstm): LSTM(100, 150, num_layers=3, batch_first=True)
  (linear): Linear(in_features=150, out_features=383, bias=True)
)

In [ ]:
loss_fn= nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(params= model.parameters(),
                            lr= 0.001)

In [ ]:
epochs= 50

for epoch in range(epochs):
  total_loss= 0

  for batch_X, batch_y in dataloader:
    batch_X, batch_y= batch_X.to(device), batch_y.to(device)

    y_pred= model(batch_X)
    loss= loss_fn(y_pred, batch_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss+= loss.item()

  print(f"Epoch: {epoch+ 1} | Loss: {total_loss:.4f}")

Epoch: 1 | Loss: 170.8622
Epoch: 2 | Loss: 158.5146
Epoch: 3 | Loss: 157.0451
Epoch: 4 | Loss: 156.7629
Epoch: 5 | Loss: 156.5612
Epoch: 6 | Loss: 156.4198
Epoch: 7 | Loss: 155.8365
Epoch: 8 | Loss: 154.6014
Epoch: 9 | Loss: 150.5576
Epoch: 10 | Loss: 143.2988
Epoch: 11 | Loss: 136.2745
Epoch: 12 | Loss: 129.0139
Epoch: 13 | Loss: 122.4029
Epoch: 14 | Loss: 116.4554
Epoch: 15 | Loss: 110.3713
Epoch: 16 | Loss: 104.8586
Epoch: 17 | Loss: 98.8458
Epoch: 18 | Loss: 93.8942
Epoch: 19 | Loss: 88.7072
Epoch: 20 | Loss: 83.9581
Epoch: 21 | Loss: 79.6484
Epoch: 22 | Loss: 74.8932
Epoch: 23 | Loss: 70.6915
Epoch: 24 | Loss: 66.6829
Epoch: 25 | Loss: 62.6636
Epoch: 26 | Loss: 59.1160
Epoch: 27 | Loss: 55.8863
Epoch: 28 | Loss: 52.6117
Epoch: 29 | Loss: 50.1307
Epoch: 30 | Loss: 46.7810
Epoch: 31 | Loss: 44.1089
Epoch: 32 | Loss: 41.0771
Epoch: 33 | Loss: 38.5842
Epoch: 34 | Loss: 36.5553
Epoch: 35 | Loss: 34.5480
Epoch: 36 | Loss: 32.4200
Epoch: 37 | Loss: 30.7022
Epoch: 38 | Loss: 29.4173
Epoch

In [ ]:
# prediction

def prediction(model, vocab, query):
  # toeknize the words
  tokenized_query= word_tokenize(query.lower())

  # text -> numerical indeces
  numerical_query= text_to_indices(tokenized_query, vocab)

  # apply padding to numerical indeces
  padded_query= torch.tensor([0]* (len_list - len(numerical_query)) + numerical_query, dtype= torch.long).unsqueeze(0)

  # send to model
  y_logits= model(padded_query)

  # predicted index
  value, index= torch.max(y_logits, dim= 1)

  # merge with text
  return query + " " + list(vocab.keys())[index]


In [ ]:
prediction(model, vocab, "You are content with my songs")

'You are content with my songs and'

In [ ]:
num_tokens= 10
input_text= "My soul"

for i in range(num_tokens):
  output= prediction(model, vocab, input_text)
  print(output)
  input_text= output

My soul fails
My soul fails to
My soul fails to reach
My soul fails to reach out
My soul fails to reach out to
My soul fails to reach out to them
My soul fails to reach out to them them
My soul fails to reach out to them them found—
My soul fails to reach out to them them found— ?
My soul fails to reach out to them them found— ? away
